In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

with open("../config.yaml") as f:
    config = yaml.safe_load(f)

csv_path = config["paths"]["fer_csv"]
csv_path


In [ ]:
.\.venv\Scripts\python.exe scripts\evaluate_model.py

In [ ]:
import os
import pandas as pd

# Resolve CSV path relative to project root (not the notebooks/ working dir)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
full_csv_path = os.path.join(repo_root, csv_path)

# Check if CSV file exists at resolved path
if not os.path.exists(full_csv_path):
    print(f"❌ File not found: {full_csv_path}")
    print("\nTo download the FER2013 dataset:")
    print("1. Visit: https://www.kaggle.com/datasets/deadskull7/fer2013")
    print("2. Download fer2013.csv")
    print(f"3. Place it in: {full_csv_path}")
    print("\nOr run: python scripts/download_data.py")
else:
    try:
        fer = pd.read_csv(full_csv_path)
        print(f"✓ Dataset loaded successfully: {fer.shape[0]} rows, {fer.shape[1]} columns")
        fer.head()
    except pd.errors.EmptyDataError:
        print(f"❌ The file exists but appears to be empty or contains no parsable columns: {full_csv_path}")
        print("Try re-downloading the dataset or opening the file to inspect its contents.")
    except Exception as e:
        print(f"❌ Error reading CSV: {e}")
        print("Check file encoding, delimiters, or file integrity.")


In [ ]:
fer.head()

In [ ]:
fer.info()
fer["Usage"].value_counts()
fer["emotion"].value_counts()


In [ ]:
emotion_map = {
    0: "Angry",
    1: "Disgust",
    2: "Fear",
    3: "Happy",
    4: "Sad",
    5: "Surprise",
    6: "Neutral",
}
fer["emotion_label"] = fer["emotion"].map(emotion_map)
fer[["emotion", "emotion_label"]].head()


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=fer, x="emotion_label", order=sorted(fer["emotion_label"].unique()))
plt.xticks(rotation=45)
plt.title("Emotion Distribution in FER2013")
plt.show()


In [ ]:
def row_to_image(pixels_str, size=48):
    arr = np.fromstring(pixels_str, dtype=int, sep=" ")
    return arr.reshape(size, size)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10,6))
axes = axes.flatten()

for idx, (emotion_id, label) in enumerate(emotion_map.items()):
    sample = fer[fer["emotion"] == emotion_id].iloc[0]
    img = row_to_image(sample["pixels"])
    axes[idx].imshow(img, cmap="gray")
    axes[idx].set_title(label)
    axes[idx].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fer["Usage"].value_counts().plot(kind="bar")
plt.title("Usage Split (Train / PublicTest / PrivateTest)")
plt.show()
